In [1]:
import numpy as np
import polars as pl
import pandas as pd
import os
import tensorflow as tf
from concurrent.futures import ThreadPoolExecutor
from queue import Queue
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from working_data import clean_cols, clean_non_minute_rows, alt_label_df as label_df, normalize_by_window
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from model_builder_trans import combined_loss
from scipy.signal import savgol_filter
from sklearn.model_selection import train_test_split


2024-10-29 15:04:55.296505: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-29 15:04:55.842008: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-29 15:04:56.986107: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2024-10-29 15:04:59.026321: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-10-29 15:04:59.220803: 

In [2]:
NORMALIZING_WINDOW_SIZE = 60*24
LABELING_WINDOW_SIZE = 20
POSITIVE_SLOPE = 0.3
LABEL_CUR_CANDLE_MULTIPLIER = 0
LABEL_MEAN_MULTIPLIER = 6
BATCH_SIZE = 32
NUM_TOKENS = 128
LOOKBACK_WINDOW = NUM_TOKENS + 1
D_MODEL = 128
FF_DIM = 256
NUM_HEADS = D_MODEL//16

In [3]:
file_paths = [
    "data/GBPUSD/minutes.csv",
    "data/EURUSD/minutes.csv",
    "data/USDJPY/minutes.csv",
    "data/USDCHF/minutes.csv"
]

working_path = "working"   

In [4]:
def split_df(
        df, 
        dump_path,
        base_name,
        train_size=0.7, 
        cols=[
    'open_normalized',
    'high_normalized',
    'low_normalized',
    'close_normalized',
    'target'
    ]):

    df = df[cols]

    train_path = os.path.join(dump_path, 'train/')
    val_path = os.path.join(dump_path, 'val/')
    test_path = os.path.join(dump_path, 'test/')
    if not os.path.exists(train_path):
        os.makedirs(train_path)
    if not os.path.exists(val_path):
        os.makedirs(val_path)
    if not os.path.exists(test_path):
        os.makedirs(test_path)

    train_data, temp_data = train_test_split(df, test_size=1-train_size, shuffle=False)
    val_data, test_data = train_test_split(temp_data, test_size=0.50, shuffle=False)
    train_data.to_csv(os.path.join(train_path, f'{base_name}.csv'), index=False)
    val_data.to_csv(os.path.join(val_path, f'{base_name}.csv'), index=False)
    test_data.to_csv(os.path.join(test_path, f'{base_name}.csv'), index=False)

In [5]:
# train_files = []
# val_files = []
# test_files = []

# for source_csv in file_paths:
#     print(source_csv)
#     base_name = source_csv.split('/')[1]
#     df = pd.read_csv(source_csv)
#     df = clean_non_minute_rows(df)
#     df = clean_cols(df)
#     break_point = len(df) - len(df)//20
#     df = df[break_point:]
#     df = normalize_by_window(
#         df, 
#         window_size=NORMALIZING_WINDOW_SIZE, 
#         normalizing_cols=[
#             'open',
#             'high',
#             'low',
#             'close',
#         ])
#     df = label_df(df, window_size=LABELING_WINDOW_SIZE, mean_multiplier=LABEL_MEAN_MULTIPLIER, cur_candle_multiplier=LABEL_CUR_CANDLE_MULTIPLIER)
#     split_df(
#         df=df, 
#         dump_path="small_data",
#         base_name=base_name,
#         cols=[
#             'open',
#             'high',
#             'low',
#             'close',
#             'open_normalized',
#             'high_normalized',
#             'low_normalized',
#             'close_normalized',
#             'target'
#         ])

In [6]:
import tensorflow as tf
import numpy as np
import polars as pl

def prepare_data(file_paths, num_tokens, window_size=1440, batch_size=32, smote=False, shuffle=False, cols=['open', 'high', 'low', 'close']):
    emd_range = window_size
    collect_cols = cols + ['target']

    df_idx_dict = {}
    df_idx_list = []

    for file_path in file_paths:
        df_lazy = pl.scan_csv(file_path).select(collect_cols)
        df_collected = df_lazy.collect()
        total_rows = df_collected.shape[0]
        df_idx_dict[file_path] = df_collected
        indices = list(range(emd_range, total_rows))
        if smote:
            df_collected = df_collected.with_columns(pl.arange(0, total_rows).alias("index"))
            indices_target_1 = df_collected.filter(
                (pl.col("target") == 1) & (pl.col("index") >= emd_range)
            ).select("index").to_series().to_list()

            # Get indices where target is 0 and >= num_tokens
            indices_target_0 = df_collected.filter(
                (pl.col("target") == 0) & (pl.col("index") >= emd_range)
            ).select("index").to_series().to_list()

            df_collected = df_collected.drop('index')

            indices_target_1_complete = []
            while len(indices_target_1_complete) < len(indices_target_0):
                indices_target_1_complete += indices_target_1

            indices_target_1_complete = indices_target_1_complete[:len(indices_target_0)]

            indices = indices_target_0 + indices_target_1_complete
        
        df_idx_list += [(file_path, idx) for idx in indices]

    if shuffle:
        np.random.shuffle(df_idx_list)
    
    while True:
        input_list = []
        target_list = []

        for df_name, idx in df_idx_list:
            df_collected = df_idx_dict[df_name]
            signal = np.array(df_collected[cols][idx - emd_range + 1:idx + 1])

            # Fetch the previous num_prev + 1 rows for the input based on the current index
            input_rows = signal[-num_tokens:]
            target_value = df_collected[idx, -1]  # Get 'target' for the target

            # Append the input rows to the input list
            input_list.append(input_rows)
            target_list.append(target_value)

            # Yield once we have enough for a batch
            if len(input_list) == batch_size:
                input_array = np.array(input_list)
                target_array = np.array(target_list)

                # Convert NumPy arrays to TensorFlow tensors
                input_tensor = tf.convert_to_tensor(input_array, dtype=tf.float32)
                target_tensor = tf.convert_to_tensor(target_array, dtype=tf.int32)

                yield input_tensor, target_tensor

                # Reset lists for the next batch

                input_list.clear()
                target_list.clear()

        break

def create_dataset_generator(file_path, batch_size, num_tokens, window_size=LOOKBACK_WINDOW, shuffle=False, repeat=False, smote=False, cols=['open', 'high', 'low', 'close']):
    dataset = tf.data.Dataset.from_generator(
        lambda: prepare_data(file_path, window_size=window_size, batch_size=batch_size, num_tokens=num_tokens, shuffle=shuffle, smote=smote, cols=cols),
        output_signature=(
            tf.TensorSpec(shape=(None, num_tokens, len(cols)), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.int32)
        )
    )
    if repeat:
        dataset = dataset.repeat()

    return dataset

In [7]:
def get_total_rows(file_paths, num_tokens, smote=False):
    # Count the total number of rows in the CSV file
    return_rows = 0
    for file_path in file_paths:
        df_lazy = pl.scan_csv(file_path)
        df_collected = df_lazy.collect()
        total_rows = df_collected.shape[0]
        if not smote:
            return_rows += total_rows
        else:
            df_collected = df_collected.with_columns(pl.arange(0, total_rows).alias("index"))
            indices_target_0 = df_collected.filter(
                    (pl.col("target") == 0) & (pl.col("index") >= num_tokens)
            )
            return_rows += len(indices_target_0) * 2
    return return_rows

In [8]:
train_paths = [os.path.join('small_data/train/', f) for f in os.listdir('small_data/train')]
val_paths = [os.path.join('small_data/val/', f) for f in os.listdir('small_data/val')]
test_paths = [os.path.join('small_data/test/', f) for f in os.listdir('small_data/test')]

cols = [
    'open_normalized',
    'high_normalized',
    'low_normalized',
    'close_normalized'
]

train_dataset = create_dataset_generator(train_paths, batch_size=BATCH_SIZE, num_tokens=NUM_TOKENS, repeat=True, shuffle=True, smote=True, cols=cols).prefetch(tf.data.AUTOTUNE)
val_dataset = create_dataset_generator(val_paths, batch_size=BATCH_SIZE, num_tokens=NUM_TOKENS, repeat=True, cols=cols).prefetch(tf.data.AUTOTUNE)
test_dataset = create_dataset_generator(test_paths, batch_size=BATCH_SIZE, num_tokens=NUM_TOKENS, cols=cols).prefetch(tf.data.AUTOTUNE)

train_steps = get_total_rows(train_paths, num_tokens=LOOKBACK_WINDOW, smote=True)//BATCH_SIZE
val_steps = get_total_rows(val_paths, num_tokens=LOOKBACK_WINDOW)//BATCH_SIZE
test_steps = get_total_rows(test_paths, num_tokens=LOOKBACK_WINDOW)//BATCH_SIZE

In [9]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, concatenate, LayerNormalization, Dropout, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.layers import MultiHeadAttention, Add, Embedding

input_length = NUM_TOKENS 

class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, maxlen, d_model):
        super(PositionalEncoding, self).__init__()
        self.pos_encoding = self.positional_encoding(maxlen, d_model)

    def positional_encoding(self, maxlen, d_model):
        positions = np.arange(maxlen)[:, np.newaxis]
        angles = np.arange(d_model)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (angles // 2)) / np.float32(d_model))
        angle_rads = positions * angle_rates

        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

        return tf.cast(angle_rads[np.newaxis, ...], dtype=tf.float32)

    def call(self, inputs):
        return inputs + self.pos_encoding[:, :tf.shape(inputs)[1], :]


class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [Dense(ff_dim, activation="relu"), Dense(embed_dim)]
        )
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)
    


In [10]:
auc =  tf.keras.metrics.AUC()
auc.reset_state()
prec = tf.keras.metrics.Precision()
prec.reset_state()

In [11]:
input_shape = (NUM_TOKENS, 4)

input_layer = Input(shape=input_shape)
x = Conv1D(filters=D_MODEL//2, kernel_size=3, activation='relu', padding="same")(input_layer)
x = Conv1D(filters=D_MODEL, kernel_size=3, activation='relu', padding="same")(x)
x = MaxPooling1D(pool_size=2, padding="same")(x)
x = Dropout(0.1)(x)
x = PositionalEncoding(x.shape[1], d_model=D_MODEL)(x)
x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM)(x, training=True)
x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM)(x, training=True)
x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM)(x, training=True)
x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM)(x, training=True)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
outputs = Dense(1, activation='sigmoid')(x)
model = Model(inputs=input_layer, outputs=outputs)

model.load_weights("working/best_cnn_trans_layer_4.keras", skip_mismatch=True)

learning_rate = 1e-3  # Adjust this value as needed
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)


# Compile the model
model.compile(
    optimizer=optimizer, 
    loss=combined_loss, 
    metrics=['accuracy',auc, prec])

model.summary()


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 4)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 128, 64)        │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 128, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding             │ (None, 64, 128)        │             0 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 64, 128)        │       593,920 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 64, 128)        │       593,920 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, 64, 128)        │       593,920 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ (None, 64, 128)        │       593,920 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,401,345 (9.16 MB)

 Trainable params: 25,536 (99.75 KB)

 Non-trainable params: 2,375,809 (9.06 MB)

In [12]:
early_stopping = EarlyStopping(monitor='val_precision_1', 
                               patience=10, # Stops if there's no improvement in precision for 5 epochs
                               mode='max', 
                               verbose=1)

model_checkpoint = ModelCheckpoint('best_cnn_trans_model.keras', 
                                   monitor='val_precision_1', 
                                   save_best_only=True, 
                                   mode='max', 
                                   verbose=1)

# model.load_weights('best_cnn_trans_model.keras')

# Train the model using the train and validation datasets
history = model.fit(
    train_dataset,
    epochs=50,
    steps_per_epoch = train_steps,
    validation_data=val_dataset,
    validation_steps=val_steps,
    callbacks=[early_stopping, model_checkpoint]
)


# Load the best model after training
model.load_weights('best_cnn_trans_model.keras')

Epoch 1/50


I0000 00:00:1730207105.124808    4508 service.cc:145] XLA service 0x7fecd0003400 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1730207105.124849    4508 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3070 Ti Laptop GPU, Compute Capability 8.6
2024-10-29 15:05:05.238693: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-10-29 15:05:05.607547: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1730207113.640931    4630 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_144', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1730207113.806515    4632 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_144', 16 bytes spill stores, 16 bytes spill loads



    1/72650 ━━━━━━━━━━━━━━━━━━━━ 450:00:40 22s/step - accuracy: 0.7812 - auc: 0.8373 - loss: 0.7086 - precision_1: 0.6667

I0000 00:00:1730207124.092587    4508 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


72647/72650 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.7434 - auc: 0.8113 - loss: 0.7339 - precision_1: 0.6957

I0000 00:00:1730209066.686728   10103 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_36', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1730209067.174145   10113 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_36', 60 bytes spill stores, 60 bytes spill loads




Epoch 1: val_precision_1 improved from -inf to 0.19441, saving model to best_cnn_trans_model.keras
72650/72650 ━━━━━━━━━━━━━━━━━━━━ 2185s 30ms/step - accuracy: 0.7434 - auc: 0.8113 - loss: 0.7339 - precision_1: 0.6957 - val_accuracy: 0.6890 - val_auc: 0.8323 - val_loss: 0.9352 - val_precision_1: 0.1944
Epoch 2/50
 2505/72650 ━━━━━━━━━━━━━━━━━━━━ 37:13 32ms/step - accuracy: 0.7490 - auc: 0.8152 - loss: 0.7332 - precision_1: 0.7023